# RAGAS reference-free context metric -- Hybrid RAG only (interactive)

Companion to `test_ragas_metrics.ipynb` (PPR + Hybrid) -- same metric, same question set, same evaluator setup, but scores the **Hybrid RAG only** pipeline (`run_hybrid_only_question.py`) instead. Run both notebooks on the same `data/eval_multihop_100_ppr.jsonl` questions and compare the two final averages directly.

Deliberately bypasses this pipeline's usual Azure OpenAI / APIM setup -- evaluates via plain OpenAI (`OPENAI_API_KEY` from `.env`) instead, using a separate credential from the rest of the app's LLM calls.

Uses the current (non-deprecated) `ragas==0.4.3` API (`ragas.metrics.collections.ContextPrecisionWithoutReference` + `ragas.llms.llm_factory(..., provider="openai", ...)`) -- see `test_ragas_metrics.ipynb`'s title cell for the full detail on why (deprecated-name mixup in ragas's own warning message, `provider="azure"` bug in `_get_instructor_client`, async-client requirement, etc.) if you need it again.

Only the reference-free precision metric is used, deliberately -- ragas has no "recall without reference" metric, and `data/eval_multihop_100_ppr.jsonl`'s `reference_answer` is being treated as unreliable here, so recall is skipped entirely.

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

CHATBOT_ROOT = Path.cwd().resolve()
for _candidate in [CHATBOT_ROOT, *CHATBOT_ROOT.parents]:
    if (_candidate / "src" / "pipeline").exists():
        CHATBOT_ROOT = _candidate
        break
REPO_ROOT = CHATBOT_ROOT.parent
if str(CHATBOT_ROOT) not in sys.path:
    sys.path.insert(0, str(CHATBOT_ROOT))
print("CHATBOT_ROOT =", CHATBOT_ROOT)

from dotenv import load_dotenv

load_dotenv(REPO_ROOT / ".env")  # populates os.environ -- OPENAI_API_KEY lives there

from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import ContextPrecisionWithoutReference

from run_hybrid_only_question import run_hybrid_only

CHATBOT_ROOT = /Users/senghok/Projects/gds_tender_test/chatbot


/opt/anaconda3/envs/gds_test/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
EVAL_MODEL = os.getenv("RAGAS_EVAL_MODEL", "gpt-4o-mini")

DEFAULT_QUESTION = (
    "How is the combined pipe rack – equipment module connected to the "
    "compressor shelter in the proposed construction sequence?"
)

QUESTIONS_PATH = REPO_ROOT / "data" / "eval_multihop_100_ppr.jsonl"
RESULTS_PATH = CHATBOT_ROOT / "domain_data" / "eval_results" / "ragas_context_precision_multihop_eval_hybrid_only.json"

In [3]:
def get_evaluator_llm():
    """Plain OpenAI (not this app's Azure/APIM setup), authenticated via
    OPENAI_API_KEY. Must be an async client -- ascore() calls
    self.llm.agenerate() internally, which raises "Cannot use agenerate()
    with a synchronous client" against a sync client."""
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise ValueError("OPENAI_API_KEY not set (checked process env and .env at repo root)")
    async_client = AsyncOpenAI(api_key=api_key)
    return llm_factory(EVAL_MODEL, provider="openai", client=async_client)


evaluator_llm = get_evaluator_llm()
print(f"evaluator_llm ready (model={EVAL_MODEL!r})")

evaluator_llm ready (model='gpt-4o-mini')


## Toy example (Eiffel Tower)

The example from the ragas docs -- validates the setup works at all before trusting it against real pipeline output.

In [ ]:
async def test_toy_example(evaluator_llm) -> None:
    print("=" * 100)
    print("Toy example (Eiffel Tower)")
    print("=" * 100)

    metric = ContextPrecisionWithoutReference(llm=evaluator_llm)
    result = await metric.ascore(
        user_input="Where is the Eiffel Tower located?",
        response="The Eiffel Tower is located in Paris.",
        retrieved_contexts=["The Eiffel Tower is located in Paris."],
    )
    print(f"ContextPrecisionWithoutReference: {result.value}")
    return result

In [ ]:
toy_result = await test_toy_example(evaluator_llm)

## Real pipeline question (Hybrid RAG only)

Runs one question through the Hybrid-RAG-only pipeline (`run_hybrid_only_question.py` -- no graph retrieval, no domain filtering) and scores its actual retrieved contexts + answer. Change `question` in the run cell and re-run to try others without rebuilding `evaluator_llm`.

In [4]:
async def test_real_question(evaluator_llm, question: str):
    print("=" * 100)
    print(f"Real pipeline question: {question}")
    print("=" * 100)

    result = run_hybrid_only(question)
    answer = result["answer"]
    contexts = [c.get("content", "") for c in result["reranked_chunks"] if c.get("content")]
    print(f"Retrieved contexts: {len(contexts)}")
    print(f"Answer: {answer.answer[:300]}")
    print()

    metric = ContextPrecisionWithoutReference(llm=evaluator_llm)
    score = await metric.ascore(
        user_input=question, response=answer.answer, retrieved_contexts=contexts,
    )
    print(f"ContextPrecisionWithoutReference: {score.value}")
    return {"answer": answer, "contexts": contexts, "score": score}

In [ ]:
question = DEFAULT_QUESTION
real_result = await test_real_question(evaluator_llm, question)

## Batch eval: all questions in `data/eval_multihop_100_ppr.jsonl`

Runs every question through the Hybrid-RAG-only pipeline and scores each with `ContextPrecisionWithoutReference`, sequentially (`rerank_chunks`' cross-encoder isn't thread-safe on this machine -- same constraint as every other pipeline eval script in `scripts/domain_eval/`). Set `LIMIT` below to a small number first to sanity-check before the full run.

In [5]:
def load_eval_questions(limit=None):
    rows = []
    with QUESTIONS_PATH.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows[:limit] if limit else rows


LIMIT = None  # set e.g. 5 for a quick smoke test before the full 86-question run
eval_questions = load_eval_questions(limit=LIMIT)
print(f"{len(eval_questions)} questions loaded from {QUESTIONS_PATH}")

86 questions loaded from /Users/senghok/Projects/gds_tender_test/data/eval_multihop_100_ppr.jsonl


In [ ]:
eval_questions[20:]

{'question': 'Which regulatory authority governs the Environment Protection Approval - Off Plot, and under what legislative framework is EPL288 (revision) issued?',
 'reference_answer': 'The Environment Protection Approval - Off Plot is governed by the Waste Management and Pollution Control Act (NT), and EPL288 (revision) is issued under the same legislative framework.',
 'question_type': 'bridge',
 'gold_chunk_ids': ['ROC_INPEX_ITT_Ichthys_Onshore_AGRU_Upgrade_and_CCS_EPC_Contract_WS33071728_ITT_Part_C_-_13_Exhibit_K_Company_Items_and_Novated_Contracts_Rev_0_20241216_pdf_size_543754_page_16_text_chunk_2',
  'ROC_INPEX_ITT_Ichthys_Onshore_AGRU_Upgrade_and_CCS_EPC_Contract_WS33071728_ITT_Part_C_-_13_Exhibit_K_Company_Items_and_Novated_Contracts_Rev_0_20241216_pdf_size_543754_page_21_text_chunk_1'],
 'gold_relation_ids': ['d3fcd58a6b82fc2495e36278eaea34f08b67bbd6',
  '23a94edaf6277068fcb963a602b569434f0bf813'],
 'gold_entity_keys': ['a0590de261e89a4895e8856a67e6eaea9071ae82',
  'b2ea971b

In [6]:
async def run_batch_ragas_eval(evaluator_llm, questions):
    """Sequential -- run_hybrid_only uses a cross-encoder reranker that isn't
    thread-safe on this machine, so questions are scored one at a time, not
    concurrently (same constraint as every other pipeline eval script in
    scripts/domain_eval/). A single question's failure is caught and logged,
    not fatal to the whole batch."""
    metric = ContextPrecisionWithoutReference(llm=evaluator_llm)
    results = []
    for i, item in enumerate(questions, 1):
        question = item["question"]
        print(f"[{i}/{len(questions)}] {question[:80]}")
        try:
            result = run_hybrid_only(question)
            answer = result["answer"]
            contexts = [c.get("content", "") for c in result["reranked_chunks"] if c.get("content")]
            score = await metric.ascore(
                user_input=question, response=answer.answer, retrieved_contexts=contexts,
            )
            results.append({
                "id": i,
                "question": question,
                "reference_answer": item.get("reference_answer"),
                "question_type": item.get("question_type"),
                "n_hops": item.get("n_hops"),
                "ok": True,
                "answer": answer.answer,
                "n_contexts": len(contexts),
                "context_precision_without_reference": score.value,
            })
            print(f"    context_precision_without_reference={score.value:.4f}")
        except Exception as exc:
            print(f"    FAILED: {type(exc).__name__}: {exc}")
            results.append({
                "id": i, "question": question, "ok": False,
                "error": f"{type(exc).__name__}: {exc}",
            })
    return results

In [11]:
batch_results = await run_batch_ragas_eval(evaluator_llm, eval_questions[20:])

[1/66] Which regulatory authority governs the Environment Protection Approval - Off Plo
    context_precision_without_reference=0.8017
[2/66] How must temporary equipment and facilities on Site be managed to ensure complia
    context_precision_without_reference=1.0000
[3/66] Where are the liquefied natural gas (LNG) processing facilities associated with 
    context_precision_without_reference=0.5927
[4/66] How are Change Orders related to the pricing of Re-measurable Works in the conte
    context_precision_without_reference=1.0000
[5/66] What financial penalty is incurred if the individual responsible for overseeing 
    context_precision_without_reference=0.7176
[6/66] What legal framework governs the RCTI payment process that the Company may elect
    context_precision_without_reference=0.7106
[7/66] What is the purpose of documents categorized as 'Category left blank' in the con
    context_precision_without_reference=1.0000
[8/66] Why is it important to supply seamless pipes and

In [ ]:
ok_results = [r for r in batch_results if r.get("ok")]
scores = [r["context_precision_without_reference"] for r in ok_results]
avg_score = sum(scores) / len(scores) if scores else None

summary = {
    "n_questions": len(batch_results),
    "n_ok": len(ok_results),
    "n_errors": len(batch_results) - len(ok_results),
    "avg_context_precision_without_reference": round(avg_score, 4) if avg_score is not None else None,
}

print("=" * 100)
print("BATCH RAGAS EVAL (HYBRID RAG ONLY) -- SUMMARY")
print("=" * 100)
for k, v in summary.items():
    print(f"{k:<45}{v}")

RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
RESULTS_PATH.write_text(
    json.dumps({"summary": summary, "results": batch_results}, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print(f"\nWrote full detail -> {RESULTS_PATH}")